# PEC model benchmark on the log catalog

Which PEC model, and which time constant, best predicts the mount's real drift?

Every catalogued tracking segment (`utility/catalog_logs.py` -> `extract_segments.py`) gives the mount's drift
`total_accum` (guide corrections + PEC applied, so PEC's own effect is removed). The driver's real `PecAxis` is
replayed **causally** on that drift -- exactly what it would have learnt at each moment -- and scored on what
PEC actually has to do:

1. **Drift left for the guider per correction interval** -- `y(t+dt) - y(t) - rate(t)*dt`: the drift still to be
   corrected after PEC's rate is applied, for guide intervals dt of 30 s, 2 min and 5 min. No reference needed,
   so this is the decision metric.
2. **Rate-tracking error** -- PEC's rate vs the actual rate (a centred local slope of the drift, only computable
   afterwards). Useful for intuition, but its best tau follows the reference window width, so it's not used to pick tau.

Everything is relative to **no PEC** (1.0); lower is better. Code and tests: `utility/pe_analysis.py`,
`tests/test_pec_rate_benchmark.py`.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
sys.path.insert(0, os.path.abspath(os.path.join('.', '..', 'driver')))
sys.path.insert(0, os.path.abspath('.'))
import importlib, pe_analysis
importlib.reload(pe_analysis)
from pe_analysis import load_segments, replay_pec_rate, true_rate, benchmark_segments, pooled_relative, PecModel

LOG_DIR = os.path.abspath(os.path.join('..', 'logs', 'archive'))
SEG_DIR = os.path.join(LOG_DIR, 'segments')     # built by analyse_pec_theta.ipynb / extract_segments.py
MIN_MINUTES = 60
GUIDE_INTERVALS = (30, 120, 300)                # s: PHD2-ish, typical plate-solve, slow plate-solve
TAUS_MIN = (1, 2, 5, 10, 15, 21, 30, 45, 60)
TEMPLATE = 'plotly_dark'

CURRENT = PecModel('current: RLS + 2 harmonics, tau 21m', 'rls', 2, 1260)
MODELS = [CURRENT,
          PecModel('RLS linear, tau 35m', 'rls', 0, 2100),
          PecModel('RLS linear, tau 7m', 'rls', 0, 420)] + \
         [PecModel(f'EMA tau {t}m', 'ema', 0, t * 60) for t in TAUS_MIN] +          [PecModel('DEMA k0.5 tau 10m', 'dema', 0, 600, k=0.5),      # lag-compensated EMA (benchmark only):
          PecModel('DEMA k1 tau 21m', 'dema', 0, 1260, k=1.0)]       # rate = e1 + k (e1 - e2)

segs = load_segments(SEG_DIR, min_minutes=MIN_MINUTES)
print(f"{len(segs)} segments (>= {MIN_MINUTES} min) from {len({s.session for s in segs})} sessions, "
      f"{sum(s.hours for s in segs):.0f} h of drift")

## 1. What the replay looks like on one segment

Top: the drift (the guide corrections + PEC applied, i.e. what the guider would have had to correct with no PEC),
with its overall quadratic trend dashed. Bottom: its rate -- the slope of the top curve (a centred local slope, so a
falling drift is a negative rate and a level drift is zero) -- and the rate each model would have applied at that
moment. A good model follows the actual rate closely, including through reversals.

In [ ]:
SEGMENT = 'alpaca.soak_Beta4.4_09_08_sg_sglog#4'     # any s.name
AXIS = 'ra'
s = next(x for x in segs if x.name == SEGMENT)
t = s.t - s.t[0]; y = getattr(s, AXIS)
show = [CURRENT, PecModel('EMA tau 2m', 'ema', 0, 120), PecModel('EMA tau 5m', 'ema', 0, 300), PecModel('EMA tau 21m', 'ema', 0, 1260)]
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.07,
                    subplot_titles=[f'{AXIS.upper()} drift (arcsec)', f'{AXIS.upper()} drift rate = slope of the drift above (arcsec/min)'])
fig.add_trace(go.Scatter(x=t / 60, y=y, name='drift', line=dict(color='white')), row=1, col=1)
fig.add_trace(go.Scatter(x=t / 60, y=np.polyval(np.polyfit(t, y, 2), t), name='quadratic trend', line=dict(color='gray', dash='dash')), row=1, col=1)
fig.add_hline(y=0, line=dict(color='gray', width=1), row=2, col=1)
fig.add_trace(go.Scatter(x=t / 60, y=true_rate(t, y, 300), name='actual rate (+-5 min)', line=dict(color='white', width=3)), row=2, col=1)
for m, c in zip(show, ('red', 'orange', 'mediumseagreen', 'royalblue')):
    fig.add_trace(go.Scatter(x=t / 60, y=replay_pec_rate(t, y, m), name=m.name, line=dict(color=c)), row=2, col=1)
fig.update_xaxes(title_text='minutes', row=2, col=1)
fig.update_layout(height=650, width=1400, template=TEMPLATE, hovermode='x unified', title=SEGMENT)
fig.show()

## 2. Benchmark across all segments

Pooled RMS relative to no PEC, separately for sync-guided segments (drift sampled at plate solves) and
pulse-guided segments (sampled every few seconds), and for all together.

In [ ]:
bench = benchmark_segments(segs, MODELS, guide_intervals_s=GUIDE_INTERVALS, ref_half_windows_s=(150, 300))
order = ['no_pec'] + [m.name for m in MODELS]
cols = [f'guide residual {dt}s' for dt in GUIDE_INTERVALS] + ['rate (ref +-2.5 min)', 'rate (ref +-5.0 min)']
tables = {}
for kind in ('sync', 'pulse', 'all'):
    g = bench if kind == 'all' else bench[bench['kind'] == kind]
    tables[kind] = pooled_relative(g).reindex(order)[cols]
    print(f"{kind}: {g['seg'].nunique()} segments")
tables['all'].round(3)

In [ ]:
tables['sync'].round(3)

In [ ]:
tables['pulse'].round(3)

## 3. Optimal EMA time constant

Drift left for the guider vs EMA tau, for each guide interval (solid: all segments; dotted: the current model).

In [ ]:
ema = [m for m in MODELS if m.mode == 'ema']
fig = go.Figure()
for dt, c in zip(GUIDE_INTERVALS, ('royalblue', 'orange', 'mediumseagreen')):
    col = f'guide residual {dt}s'
    fig.add_trace(go.Scatter(x=[m.tau_s / 60 for m in ema], y=[tables['all'].loc[m.name, col] for m in ema],
                             mode='lines+markers', name=f'EMA, {dt} s interval', line=dict(color=c)))
    fig.add_hline(y=tables['all'].loc[CURRENT.name, col], line=dict(color=c, dash='dot'))
fig.add_hline(y=1.0, line=dict(color='gray'))
fig.update_xaxes(type='log', title='EMA tau (minutes)')
fig.update_yaxes(title='drift left for the guider, relative to no PEC')
fig.update_layout(height=500, width=1000, template=TEMPLATE, title='EMA tau sweep (dotted: current RLS + harmonics)')
fig.show()

### Per-segment best tau, and the cost of one global tau

Each segment's best tau for the 2-minute interval, and how much worse one fixed tau is than each segment's own best.

In [ ]:
INTERVAL = 120
g = bench[(bench['metric'] == f'guide residual {INTERVAL}s') & bench['model'].isin([m.name for m in ema])].copy()
g['tau_min'] = g['model'].map({m.name: m.tau_s / 60 for m in ema})
per_seg = g.groupby(['seg', 'tau_min'])['value'].apply(lambda v: np.sqrt((v ** 2).mean())).reset_index()
best = per_seg.loc[per_seg.groupby('seg')['value'].idxmin()].set_index('seg')
fig = go.Figure(go.Histogram(x=best['tau_min'].astype(str), marker_color='orange'))
fig.update_xaxes(title='best EMA tau (min)', categoryorder='array', categoryarray=[str(float(t)) for t in TAUS_MIN])
fig.update_layout(height=400, width=800, template=TEMPLATE, title=f'Best EMA tau per segment ({INTERVAL} s interval)')
fig.show()
loss = pd.DataFrame({f'{tau} min': (per_seg[per_seg['tau_min'] == tau].set_index('seg')['value'] / best['value'] - 1) * 100
                     for tau in TAUS_MIN})
loss.describe().loc[['50%', 'mean', 'max']].rename(index={'50%': 'median loss %', 'mean': 'mean loss %', 'max': 'worst loss %'}).round(1)

## 4. Findings (2026-10-01)

- **The current model (RLS + 2 harmonics, tau 21 min) leaves about as much drift for the guider as no PEC at all**
  on sync-guided segments (1.05-1.11 x), and only 0.80-0.89 x on the 4 pulse-guided ones -- behind every rate-only model.
  The harmonics add periodic predictions that are wrong as often as right.
- **Rate-only models cut the drift left for the guider by ~20-40%** depending on the interval (more for longer intervals,
  i.e. sync guiding). EMA and RLS-linear are close; EMA with a shorter tau is best.
- **Optimal EMA tau is about 5-10 minutes**, with a broad minimum. Per segment the best tau is mostly 2-5 min; a
  single global tau of 5 min loses ~3% median (worst ~17%) against each segment's own best, while 21 min loses ~10%
  median (worst ~140%).
- **Lag compensation doesn't help.** A double EMA (DEMA: `rate = e1 + k (e1 - e2)`, which cancels EMA's lag on a
  steadily changing rate) only helps at long tau (21-30 min), bringing it to about where plain EMA already is at
  5-10 min; at short tau it adds noise and does worse. Part of the visible lag in section 1 is unavoidable: the
  "actual rate" is a centred slope that uses the next 5 minutes, which no causal estimator can. No cycle is
  consistent enough to anticipate reversals (harmonic terms and DEMA would beat plain EMA if one were).
- Sync- and pulse-guided segments agree (only 4 pulse-guided segments, so the pulse result is less certain).

Caveats: drift is `total_accum` as logged, so measurement noise and step changes at syncs are included (as they are
for the real PEC). The pulse-guided result rests on 4 segments. This scores PEC's rate in isolation; the twin
(`tests/test_pec_twin_drift.py`) checks the closed loop with the PID and guider.